# Formal-Wear Detector — Training Notebook (YOLO26)

Trains an object detector on a filtered **Fashionpedia** subset for 8 garment classes, then hands you a `best.pt` to run locally.

**How to use:**
1. `Runtime -> Change runtime type -> GPU (T4)`  ← do this first!
2. `Runtime -> Run all`
3. Wait ~20-40 min. At the end, `best.pt` downloads automatically.
4. Send that `best.pt` back to Claude.

Classes: `shirt, t-shirt, pants, shorts, tie, shoe, belt, jacket`

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Install dependencies

In [ ]:
!pip install -q -U ultralytics datasets
import ultralytics; ultralytics.checks()

## 3. Config

`SRC_TO_NEW` maps Fashionpedia's category ids to our 8-class scheme. `N_TRAIN` / `N_VAL` control the subset size (kept small for fast training).

In [ ]:
# our class order (index -> name)
CLASS_NAMES = ['shirt', 't-shirt', 'pants', 'shorts', 'tie', 'shoe', 'belt', 'jacket']

# Fashionpedia source id -> our new id
SRC_TO_NEW = {
    0: 0,   # shirt, blouse   -> shirt
    1: 1,   # top, t-shirt    -> t-shirt
    6: 2,   # pants           -> pants
    7: 3,   # shorts          -> shorts
    16: 4,  # tie             -> tie
    23: 5,  # shoe            -> shoe
    19: 6,  # belt            -> belt
    4: 7,   # jacket          -> jacket
}

N_TRAIN = 2500   # images (that contain >=1 of our classes)
N_VAL   = 300

DS_ROOT = '/content/formal_ds'
MODEL   = 'yolo26n.pt'   # if this errors (too-new weights), change to 'yolo11n.pt'
EPOCHS  = 50
IMGSZ   = 640

## 4. Build the YOLO-format dataset (streaming)

We **stream** Fashionpedia (no full multi-GB download) and keep only images that contain at least one of our 8 classes. Boxes are converted from Pascal VOC `[xmin,ymin,xmax,ymax]` (absolute) to YOLO `[x_center, y_center, w, h]` (normalized).

In [ ]:
import os
from datasets import load_dataset

def build_split(split, max_images):
    img_dir = f'{DS_ROOT}/images/{split}'
    lbl_dir = f'{DS_ROOT}/labels/{split}'
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)
    ds = load_dataset('detection-datasets/fashionpedia', split=split, streaming=True)
    kept = 0
    for ex in ds:
        W, H = ex['width'], ex['height']
        objs = ex['objects']
        lines = []
        for cat, box in zip(objs['category'], objs['bbox']):
            if cat not in SRC_TO_NEW:
                continue
            xmin, ymin, xmax, ymax = box
            xc = ((xmin + xmax) / 2) / W
            yc = ((ymin + ymax) / 2) / H
            bw = (xmax - xmin) / W
            bh = (ymax - ymin) / H
            if bw <= 0 or bh <= 0:
                continue
            xc, yc = min(max(xc, 0), 1), min(max(yc, 0), 1)
            bw, bh = min(max(bw, 0), 1), min(max(bh, 0), 1)
            lines.append(f'{SRC_TO_NEW[cat]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')
        if not lines:
            continue
        name = str(ex['image_id'])
        ex['image'].convert('RGB').save(f'{img_dir}/{name}.jpg', quality=90)
        with open(f'{lbl_dir}/{name}.txt', 'w') as f:
            f.write('\n'.join(lines))
        kept += 1
        if kept % 250 == 0:
            print(f'  {split}: {kept}/{max_images}')
        if kept >= max_images:
            break
    print(f'{split} done -> {kept} images')
    return kept

build_split('train', N_TRAIN)
build_split('val', N_VAL)

## 5. Write `data.yaml`

In [ ]:
names_block = '\n'.join(f'  {i}: {n}' for i, n in enumerate(CLASS_NAMES))
yaml_txt = f"""path: {DS_ROOT}
train: images/train
val: images/val
names:
{names_block}
"""
with open(f'{DS_ROOT}/data.yaml', 'w') as f:
    f.write(yaml_txt)
print(yaml_txt)

## 6. Sanity check — draw boxes on a few images

**Look at these!** The boxes should sit correctly on garments. If they look wrong/shifted, stop — the conversion is off and training would be wasted.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import glob, random

samples = random.sample(glob.glob(f'{DS_ROOT}/images/train/*.jpg'), 4)
fig, axes = plt.subplots(1, 4, figsize=(20, 6))
for ax, imgp in zip(axes, samples):
    img = Image.open(imgp); W, H = img.size
    ax.imshow(img); ax.axis('off')
    lblp = imgp.replace('/images/', '/labels/').replace('.jpg', '.txt')
    for line in open(lblp):
        c, xc, yc, bw, bh = line.split()
        c = int(c); xc, yc, bw, bh = float(xc)*W, float(yc)*H, float(bw)*W, float(bh)*H
        ax.add_patch(patches.Rectangle((xc-bw/2, yc-bh/2), bw, bh, fill=False, color='lime', lw=2))
        ax.text(xc-bw/2, yc-bh/2-4, CLASS_NAMES[c], color='black', fontsize=9,
                bbox=dict(facecolor='lime', edgecolor='none', pad=1))
plt.tight_layout(); plt.show()

## 7. Train YOLO26

~20-40 min on a T4. `patience=15` early-stops if it stops improving.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
results = model.train(
    data=f'{DS_ROOT}/data.yaml',
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=16,
    patience=15,
    name='formal_detector',
)

## 8. Validate + preview predictions

In [ ]:
metrics = model.val()
print('mAP50-95:', round(metrics.box.map, 4))
print('mAP50   :', round(metrics.box.map50, 4))

In [ ]:
import glob, random
from PIL import Image
import matplotlib.pyplot as plt

val_imgs = random.sample(glob.glob(f'{DS_ROOT}/images/val/*.jpg'), 4)
preds = model.predict(val_imgs, conf=0.3, verbose=False)
fig, axes = plt.subplots(1, 4, figsize=(20, 6))
for ax, p in zip(axes, preds):
    ax.imshow(Image.fromarray(p.plot()[..., ::-1])); ax.axis('off')
plt.tight_layout(); plt.show()

## 9. Download `best.pt`

This is the only file you need to send back.

In [ ]:
import shutil
from google.colab import files

best = 'runs/detect/formal_detector/weights/best.pt'
shutil.copy(best, 'best.pt')
print('size:', round(os.path.getsize('best.pt')/1e6, 1), 'MB')
files.download('best.pt')